In [1]:
import pathlib
import pandas as pd


def load_csv(filename):
    """Return the first existing path for a given CSV filename."""
    possible_paths = [
        pathlib.Path("../input/osic-pulmonary-fibrosis-progression") / filename,
        pathlib.Path("data/osic-pulmonary-fibrosis-progression") / filename,
        pathlib.Path("data") / filename,
        pathlib.Path(filename),
    ]
    for p in possible_paths:
        if p.is_file():
            return pd.read_csv(p)
    raise FileNotFoundError(
        f"Could not find {filename} in any of the expected locations."
    )


train = load_csv("train.csv")
test = load_csv("test.csv")



In [2]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "FVC"
FEATURES = ["Weeks", "Percent", "Age", "Sex", "SmokingStatus"]
CONF_MIN = 70  # lower bound for confidence as per competition definition
# Increase the constant confidence to improve the Laplace‑Log‑Likelihood score
CONST_CONF = 200.0  # >= CONF_MIN, chosen to move score toward the target

X = train[FEATURES]
y = train[TARGET]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ["Weeks", "Percent", "Age"]
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_features = ["Sex", "SmokingStatus"]
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=5,
    max_depth=None,
)

pipeline = Pipeline(steps=[("preprocess", preprocess), ("rf", rf)])

pipeline.fit(X_train, y_train)


def confidence(pipeline, X):
    """
    Return a safe constant confidence for every sample.
    This guarantees the confidence is >= CONF_MIN and eliminates NaNs.
    """
    n_samples = X.shape[0]
    conf = np.full(n_samples, CONST_CONF, dtype=float)
    conf = np.clip(conf, CONF_MIN, None)
    return conf


def laplace_log_likelihood(
    actual_fvc, predicted_fvc, confidence_vals, return_values=False
):
    sd_clipped = np.maximum(confidence_vals, CONF_MIN)
    delta = np.minimum(np.abs(actual_fvc - predicted_fvc), 1000)
    metric = -np.sqrt(2) * delta / sd_clipped - np.log(np.sqrt(2) * sd_clipped)
    return np.mean(metric) if not return_values else metric


train_pred = pipeline.predict(X_train)
val_pred = pipeline.predict(X_val)

train_conf = confidence(pipeline, X_train)
val_conf = confidence(pipeline, X_val)

print("Train OSCI score: ", laplace_log_likelihood(y_train, train_pred, train_conf))
print("Val   OSCI score: ", laplace_log_likelihood(y_val, val_pred, val_conf))

print("Train RMSE score: ", np.sqrt(mean_squared_error(y_train, train_pred)))
print("Val   RMSE score: ", np.sqrt(mean_squared_error(y_val, val_pred)))

test_features = test[FEATURES]
test_pred = pipeline.predict(test_features)
test_conf = confidence(pipeline, test_features)

submission = pd.DataFrame(
    {
        "Patient_Week": test["Patient"].astype(str) + "_" + test["Weeks"].astype(str),
        "FVC": test_pred,
        "Confidence": test_conf,
    }
)

submission = submission[["Patient_Week", "FVC", "Confidence"]]

output_path = pathlib.Path("submission.csv")
submission.to_csv(output_path, index=False)
print(f"Submission written to {output_path.resolve()}")

Train OSCI score:  -6.025761867350918
Val   OSCI score:  -6.62811097354458
Train RMSE score:  78.59513434045786
Val   RMSE score:  203.68896487009172
Submission written to /kaggle/working/submission.csv
